In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("activity_name", "")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
activity_name = dbutils.widgets.get("activity_name")

In [0]:
dbutils.widgets.text("run_date", "")
dbutils.widgets.text("days_back", "30")
dbutils.widgets.text("environment", "dev")

run_date = dbutils.widgets.get("run_date")
days_back = int(dbutils.widgets.get("days_back"))
environment = dbutils.widgets.get("environment")

print(f"Run date: {run_date}")
print(f"Days back: {days_back}")
print(f"Environment: {environment}")

Run date: 
Days back: 30
Environment: dev


In [0]:
# ------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

In [0]:
# ------------------------------------------
# 2. CONNECT TO ADLS USING KEY VAULT SECRET
# ------------------------------------------

storage_account_name = "staihospitaldev001"

storage_account_key = dbutils.secrets.get(
    scope="kv-aihoispital-scope",
    key="kv-aihospital-dev"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_path   = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

print("Bronze path:", bronze_path)
print("Silver path:", silver_path)
print("Gold path:", gold_path)

Bronze path: abfss://bronze@staihospitaldev001.dfs.core.windows.net/
Silver path: abfss://silver@staihospitaldev001.dfs.core.windows.net/
Gold path: abfss://gold@staihospitaldev001.dfs.core.windows.net/


In [0]:
from pyspark.sql import Row
from datetime import datetime

pipeline_run_id = dbutils.widgets.get("pipeline_run_id") if "pipeline_run_id" else ""
activity_name = dbutils.widgets.get("activity_name") if "activity_name" else ""

log_path = f"{gold_path}pipeline_logs"

def write_pipeline_log(stage_name, status, records_processed=0, message=""):
    log_df = spark.createDataFrame([
        Row(
            pipeline_run_id=pipeline_run_id,
            activity_name=activity_name,
            stage_name=stage_name,
            status=status,
            records_processed=int(records_processed),
            message=message,
            log_timestamp=datetime.utcnow()
        )
    ])
    
    log_df.write.format("delta").mode("append").save(log_path)

In [0]:
from pyspark.sql import Row
from datetime import datetime

# Widgets from ADF
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("activity_name", "")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
activity_name = dbutils.widgets.get("activity_name")

log_path = f"{gold_path}pipeline_logs"

def write_pipeline_log(stage_name, status, records_processed=0, message=""):
    log_df = spark.createDataFrame([
        Row(
            pipeline_run_id=pipeline_run_id,
            activity_name=activity_name,
            stage_name=stage_name,
            status=status,
            records_processed=int(records_processed),
            message=message,
            log_timestamp=datetime.utcnow()
        )
    ])
    
    log_df.write.format("delta").mode("append").save(log_path)

In [0]:
write_pipeline_log(
    stage_name="Gold Feature Engineering",
    status="Started",
    records_processed=0,
    message="Gold fact and ML feature engineering started"
)

In [0]:
# ------------------------------------------
# 3. READ SILVER TABLES
# ------------------------------------------

silver_admissions_enriched = spark.read.format("delta").load(silver_path + "admissions_enriched")
silver_incidents_enriched  = spark.read.format("delta").load(silver_path + "incidents_enriched")
silver_staff_activity      = spark.read.format("delta").load(silver_path + "staff_activity")

print("Admissions:", silver_admissions_enriched.count())
print("Incidents:", silver_incidents_enriched.count())
print("Staff activity:", silver_staff_activity.count())

Admissions: 1147
Incidents: 1285
Staff activity: 16374


In [0]:
# ------------------------------------------
# 4. BUILD INCIDENT METRICS PER ADMISSION
# ------------------------------------------

incident_metrics = (
    silver_incidents_enriched
    .groupBy("AdmissionID")
    .agg(
        F.count("*").alias("IncidentCount"),
        F.sum("EscalatedFlag").alias("EscalatedIncidentCount"),
        F.avg("SeverityScore").alias("AvgSeverityScore"),
        F.max("SeverityScore").alias("MaxSeverityScore")
    )
)

In [0]:
# ------------------------------------------
# 5. BUILD STAFFING METRICS PER WARD
# ------------------------------------------

staffing_metrics = (
    silver_staff_activity
    .groupBy("WardID", "WardName")
    .agg(
        F.countDistinct("ShiftID").alias("ShiftCoverageCount"),
        F.countDistinct("StaffID").alias("DistinctStaffCount"),
        F.sum("HoursPlanned").alias("TotalPlannedHours"),
        F.sum(F.when(F.col("ShiftType") == "Night", 1).otherwise(0)).alias("NightShiftCount")
    )
)

In [0]:
# ------------------------------------------
# 6. CREATE GOLD FACT PATIENT SAFETY
# ------------------------------------------

gold_fact_patient_safety = (
    silver_admissions_enriched.alias("a")
    .join(
        incident_metrics.alias("i"),
        F.col("a.AdmissionID") == F.col("i.AdmissionID"),
        "left"
    )
    .join(
        staffing_metrics.alias("s"),
        F.col("a.WardID") == F.col("s.WardID"),
        "left"
    )
    .select(
        F.col("a.AdmissionID"),
        F.col("a.PatientID"),
        F.col("a.FirstName"),
        F.col("a.LastName"),
        F.col("a.PatientAge"),
        F.col("a.Gender"),
        F.col("a.RiskCategory"),
        F.col("a.WardID"),
        F.col("a.WardName"),
        F.col("a.Specialty"),
        F.col("a.Capacity"),
        F.col("a.AdmissionDate"),
        F.col("a.DischargeDate"),
        F.col("a.AdmissionType"),
        F.col("a.LengthOfStayDays"),
        F.col("a.AdmissionStatus"),
        F.col("a.LengthOfStayBand"),
        F.coalesce(F.col("i.IncidentCount"), F.lit(0)).alias("IncidentCount"),
        F.coalesce(F.col("i.EscalatedIncidentCount"), F.lit(0)).alias("EscalatedIncidentCount"),
        F.round(F.coalesce(F.col("i.AvgSeverityScore"), F.lit(0.0)), 2).alias("AvgSeverityScore"),
        F.coalesce(F.col("i.MaxSeverityScore"), F.lit(0)).alias("MaxSeverityScore"),
        F.coalesce(F.col("s.ShiftCoverageCount"), F.lit(0)).alias("ShiftCoverageCount"),
        F.coalesce(F.col("s.DistinctStaffCount"), F.lit(0)).alias("DistinctStaffCount"),
        F.coalesce(F.col("s.TotalPlannedHours"), F.lit(0)).alias("TotalPlannedHours"),
        F.coalesce(F.col("s.NightShiftCount"), F.lit(0)).alias("NightShiftCount")
    )
)

In [0]:
# ------------------------------------------
# 7. ADD BI-READY DERIVED COLUMNS
# ------------------------------------------

gold_fact_patient_safety = (
    gold_fact_patient_safety
    .withColumn(
        "HasIncidentFlag",
        F.when(F.col("IncidentCount") > 0, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "HasEscalatedIncidentFlag",
        F.when(F.col("EscalatedIncidentCount") > 0, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IncidentRatePerDay",
        F.when(F.col("LengthOfStayDays") > 0, F.round(F.col("IncidentCount") / F.col("LengthOfStayDays"), 4))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "StaffingPressureFlag",
        F.when((F.col("DistinctStaffCount") < 5) | (F.col("NightShiftCount") < 2), F.lit(1))
         .otherwise(F.lit(0))
    )
)

In [0]:
# ------------------------------------------
# 10. CREATE GOLD PATIENT INCIDENT FEATURES
# ------------------------------------------

gold_patient_incident_features = (
    gold_fact_patient_safety
    .select(
        "AdmissionID",
        "PatientID",
        "PatientAge",
        "Gender",
        "RiskCategory",
        "WardID",
        "WardName",
        "Specialty",
        "Capacity",
        "AdmissionType",
        "LengthOfStayDays",
        "AdmissionStatus",
        "LengthOfStayBand",
        "IncidentCount",
        "EscalatedIncidentCount",
        "AvgSeverityScore",
        "MaxSeverityScore",
        "ShiftCoverageCount",
        "DistinctStaffCount",
        "TotalPlannedHours",
        "NightShiftCount",
        "StaffingPressureFlag"
    )
)

In [0]:
# ------------------------------------------
# 11. ADD ML TARGET COLUMN
# ------------------------------------------

gold_patient_incident_features = (
    gold_patient_incident_features
    .withColumn(
        "WillIncidentOccur",
        F.when(F.col("IncidentCount") > 0, F.lit(1)).otherwise(F.lit(0))
    )
)

In [0]:
# ------------------------------------------
# 12. ADD DERIVED ML FEATURES
# ------------------------------------------

gold_patient_incident_features = (
    gold_patient_incident_features
    .withColumn(
        "IsHighRiskPatient",
        F.when(F.col("RiskCategory") == "High", F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IsEmergencyAdmission",
        F.when(F.col("AdmissionType") == "Emergency", F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IsActiveAdmission",
        F.when(F.col("AdmissionStatus") == "Active", F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "AvgHoursPerStaff",
        F.when(F.col("DistinctStaffCount") > 0,
               F.round(F.col("TotalPlannedHours") / F.col("DistinctStaffCount"), 2))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "IncidentRatePerDay",
        F.when(F.col("LengthOfStayDays") > 0,
               F.round(F.col("IncidentCount") / F.col("LengthOfStayDays"), 4))
         .otherwise(F.lit(0.0))
    )
)

In [0]:
# ------------------------------------------
# 13. ADD SIMPLE CATEGORICAL ML HELPERS
# ------------------------------------------

gold_patient_incident_features = (
    gold_patient_incident_features
    .withColumn(
        "GenderCode",
        F.when(F.col("Gender") == "Male", F.lit(1))
         .when(F.col("Gender") == "Female", F.lit(0))
         .otherwise(F.lit(-1))
    )
    .withColumn(
        "RiskCategoryCode",
        F.when(F.col("RiskCategory") == "Low", F.lit(1))
         .when(F.col("RiskCategory") == "Medium", F.lit(2))
         .when(F.col("RiskCategory") == "High", F.lit(3))
         .otherwise(F.lit(0))
    )
)

In [0]:
# ------------------------------------------
# 14. WRITE GOLD ML FEATURE TABLE
# ------------------------------------------

gold_patient_incident_features.write.format("delta").mode("overwrite").save(
    gold_path + "patient_incident_features"
)

print("gold_patient_incident_features written successfully")

gold_patient_incident_features written successfully


In [0]:
# ------------------------------------------
# 15. VALIDATE GOLD ML FEATURE TABLE
# ------------------------------------------

gold_ml_check = spark.read.format("delta").load(gold_path + "patient_incident_features")

print("Gold patient incident features row count:", gold_ml_check.count())

display(
    gold_ml_check.select(
        "AdmissionID",
        "PatientID",
        "PatientAge",
        "Gender",
        "RiskCategory",
        "AdmissionType",
        "WardName",
        "LengthOfStayDays",
        "IncidentCount",
        "EscalatedIncidentCount",
        "DistinctStaffCount",
        "TotalPlannedHours",
        "StaffingPressureFlag",
        "WillIncidentOccur"
    )
)

Gold patient incident features row count: 1147


AdmissionID,PatientID,PatientAge,Gender,RiskCategory,AdmissionType,WardName,LengthOfStayDays,IncidentCount,EscalatedIncidentCount,DistinctStaffCount,TotalPlannedHours,StaffingPressureFlag,WillIncidentOccur
700000,10000,44,Male,Low,Planned,Rehab & Recovery Ward,185,3,1,220,25684,0,1
700001,10001,36,Female,Low,Transfer,Older Adults Ward,18,0,0,220,31228,0,0
700002,10001,36,Female,Low,Transfer,Secure Unit,160,0,0,220,28576,0,0
700003,10002,47,Male,Medium,Emergency,Rehab & Recovery Ward,315,0,0,220,25684,0,0
700004,10002,47,Male,Medium,Emergency,Acute Psychiatric Ward,13,1,0,220,46940,0,1
700005,10003,59,Male,Medium,Planned,Rehab & Recovery Ward,20,0,0,220,25684,0,0
700006,10003,59,Male,Medium,Emergency,Older Adults Ward,13,1,1,220,31228,0,1
700007,10003,59,Male,Medium,Emergency,Rehab & Recovery Ward,288,0,0,220,25684,0,0
700008,10004,34,Male,Low,Emergency,Older Adults Ward,294,0,0,220,31228,0,0
700009,10005,34,Male,Low,Planned,Older Adults Ward,60,0,0,220,31228,0,0


In [0]:
sql_password = dbutils.secrets.get(
    scope="kv-aihospital-sql-scope",
    key="azure-sql-password"
)

print("Password loaded:", bool(sql_password), "Length:", len(sql_password))

Password loaded: True Length: 13


In [0]:
# ------------------------------------------
# AZURE SQL CONNECTION
# ------------------------------------------

sql_scope = "kv-aihospital-sql-scope"  
sql_server = "aihospital-sql-dev.database.windows.net"
sql_database = "aihospital-sql-dev"
sql_username = "aihospital-sql-dev"

sql_password = dbutils.secrets.get(
    scope=sql_scope,
    key="azure-sql-password"
)

jdbc_url = (
    f"jdbc:sqlserver://{sql_server}:1433;"
    f"database={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": sql_username,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("Server:", sql_server)
print("Database:", sql_database)
print("Username:", sql_username)

Server: aihospital-sql-dev.database.windows.net
Database: aihospital-sql-dev
Username: aihospital-sql-dev


In [0]:
# ------------------------------------------
# FUNCTION TO WRITE FACT TABLE TO AZURE SQL
# ------------------------------------------
def write_to_azure_sql(df, table_name, mode="overwrite"):
    df.write.jdbc(
        url=jdbc_url,
        table=table_name,
        mode=mode,
        properties=connection_properties
    )
    print(f"Write to {table_name} completed successfully.")

In [0]:
# ------------------------------------------
# WRITE FACT TABLE TO AZURE SQL
# ------------------------------------------

write_to_azure_sql(
    gold_fact_patient_safety,
    "dbo.fact_patient_safety",
    mode="overwrite"
)

Write to dbo.fact_patient_safety completed successfully.


In [0]:
gold_record_count = gold_fact_patient_safety.count()

write_pipeline_log(
    stage_name="Gold Feature Engineering",
    status="Succeeded",
    records_processed=gold_record_count,
    message="Gold fact table and patient incident feature table completed successfully"
)